## SIM - Sistema de Informação sobre Mortalidade: DATASUS

SIM 2020–2024 -> Silver. pandas + pyarrow; sem acesso ao PostgreSQL.
Preserva todas as linhas e a união das colunas originais (texto, nomes minúsculos).
DTOBITO/DTNASC originais são mantidos; datas derivadas são acrescentadas.
Demais datas, idade codificada e causas permanecem texto, sem interpretação clínica.
Não filtra COVID, não remove duplicatas, não infere identidade de pessoas.
Chave técnica proposta: ano_arquivo + contador, restrita à coleta selecionada.
Referência populacional pelo ano do óbito; municípios ligados por código exato.
Não aplica automaticamente as equivalências observadas em outra base (SRAG).

In [2]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv
import hashlib
import json
import sqlite3
import uuid

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [3]:
PROJETO = Path(r"D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil")
COLETA = PROJETO / "Camada_Bronze/dados_mortalidade_sim/coleta_20261023_103728_a8cccf84"
SILVER = PROJETO / "Camada_Silver"
IBGE = SILVER / "dados/populacao_ibge/20261024_163052_e1db3d2b/populacao_municipios_2020_2024.parquet"
ANOS = tuple(range(2020, 2025))
ESPERADOS = {2020: 1556824, 2021: 1832649, 2022: 1544266, 2023: 1465610, 2024: 1532015}
LOTE = 50_000
PAPEIS = {"residencia": "codmunres", "ocorrencia": "codmunocor"}

In [4]:
def texto(s):
    return s.astype("string").str.strip().replace("", pd.NA)


def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()


def converter_data(s):
    import re
    from datetime import datetime

    v = texto(s)

    minimo = datetime(1677, 9, 22)
    maximo = datetime(2262, 4, 11)

    curto = re.compile(r"[0-9]{8}")

    longo = re.compile(
        r"([0-9]{4})/([0-9]{2})/([0-9]{2})"
        r" ([0-9]{2}):([0-9]{2}):([0-9]{2})"
        r"(?:\.[0-9]{1,9})?"
    )

    def interpretar(valor):
        try:
            if curto.fullmatch(valor):
                data = datetime.strptime(valor, "%d%m%Y")

            else:
                partes = longo.fullmatch(valor)

                if partes is None:
                    return None

                ano, mes, dia, hora, minuto, segundo = map(
                    int, partes.groups()
                )

                # Valida tanto o calendário quanto o horário.
                datetime(ano, mes, dia, hora, minuto, segundo)

                # Preserva o dia informado, sem conversão de fuso.
                data = datetime(ano, mes, dia)

            if not minimo <= data <= maximo:
                return None

            return data

        except (ValueError, OverflowError):
            return None

    conversoes = {
        valor: interpretar(valor)
        for valor in v.dropna().unique()
    }

    datas = pd.Series(
        [
            conversoes.get(valor) if pd.notna(valor) else None
            for valor in v
        ],
        index=s.index,
        dtype="datetime64[ns]",
    )

    status = pd.Series(
        "VALIDA",
        index=s.index,
        dtype="string",
    )

    status.loc[datas.isna()] = "INVALIDA_OU_FORMATO_NAO_PREVISTO"
    status.loc[v.isna()] = "AUSENTE"

    return datas, status


def carregar_referencia():
    ref = pd.read_parquet(IBGE, columns=["ano_analise", "codigo_ibge_7", "uf", "municipio", "populacao"])
    ref["ano_analise"] = pd.to_numeric(ref.ano_analise, errors="raise").astype("Int64")
    ref = ref.loc[ref.ano_analise.isin(ANOS)].copy()
    for c in ["codigo_ibge_7", "uf", "municipio"]:
        ref[c] = texto(ref[c])
    if set(ref.ano_analise) != set(ANOS):
        raise ValueError("Referência IBGE não cobre todos os anos.")
    if not ref.codigo_ibge_7.str.fullmatch(r"[0-9]{7}", na=False).all():
        raise ValueError("Código IBGE inválido na referência.")
    if ref[["uf", "municipio"]].isna().any().any():
        raise ValueError("Nome ou UF ausente na referência IBGE.")
    ref["codigo_6"] = ref.codigo_ibge_7.str[:6]
    if ref.duplicated(["ano_analise", "codigo_6"]).any():
        raise ValueError("Código municipal/ano ambíguo na referência IBGE.")
    return ref


def vincular(df, ref, papel, campo):
    b = pd.DataFrame({"ano_analise": df.ano_analise, "codigo_6": texto(df[campo])})
    m = b.merge(ref, on=["ano_analise", "codigo_6"], how="left", sort=False, validate="many_to_one")
    status = pd.Series("SEM_CORRESPONDENCIA", index=m.index, dtype="string")
    achou = m.codigo_ibge_7.notna()
    status.loc[achou] = "VINCULADO_COM_POPULACAO"
    status.loc[achou & m.populacao.isna()] = "VINCULADO_SEM_POPULACAO"
    status.loc[~m.codigo_6.str.fullmatch(r"[0-9]{6}", na=False)] = "CODIGO_INVALIDO"
    status.loc[m.codigo_6.isna()] = "CODIGO_AUSENTE"
    status.loc[~m.ano_analise.isin(ANOS)] = "ANO_FORA_REFERENCIA"
    status.loc[m.ano_analise.isna()] = "ANO_AUSENTE"
    valido = status.isin(["VINCULADO_COM_POPULACAO", "VINCULADO_SEM_POPULACAO"])
    for destino, origem in [("codigo_ibge_7", "codigo_ibge_7"), ("municipio_ibge", "municipio"), ("uf_ibge", "uf")]:
        df[f"{destino}_{papel}"] = m[origem].where(valido).astype("string")
    df[f"status_vinculo_{papel}"] = status
    return status


def main():
    print("Iniciando tratamento SIM — verificando a coleta selecionada...", flush=True)
    fontes = {ano: COLETA / "parquet" / f"sim_{ano}.parquet" for ano in ANOS}
    for p in [*fontes.values(), IBGE]:
        if not p.is_file():
            raise FileNotFoundError(p)
    ref = carregar_referencia()
    campos = set()
    originais = {}
    for ano, p in fontes.items():
        with pq.ParquetFile(p) as pf:
            if pf.metadata.num_rows != ESPERADOS[ano]:
                raise ValueError(f"{ano}: contagem diferente da coleta revisada. Esperado {ESPERADOS[ano]:,}; encontrado {pf.metadata.num_rows:,}.")
            nomes = [n.lower() for n in pf.schema_arrow.names]
            if len(nomes) != len(set(nomes)):
                raise ValueError(f"{ano}: colisão de nomes após conversão para minúsculas.")
            if not all(pa.types.is_string(f.type) or pa.types.is_large_string(f.type) for f in pf.schema_arrow):
                raise ValueError(f"{ano}: tipo de entrada diferente de texto.")
            if not {"contador", "dtobito", "dtnasc", "codmunres", "codmunocor", "causabas"}.issubset(nomes):
                raise ValueError(f"{ano}: faltam campos obrigatórios.")
            originais[ano] = nomes
            campos.update(nomes)
    campos = sorted(campos)
    extras = [pa.field("ano_arquivo", pa.int16()), pa.field("ano_analise", pa.int16()),
              pa.field("linha_arquivo", pa.int64()), pa.field("data_obito", pa.date32()),
              pa.field("data_nascimento", pa.date32()), pa.field("status_data_obito", pa.string()),
              pa.field("status_data_nascimento", pa.string()),
              pa.field("fl_obito_ano_diferente_arquivo", pa.bool_()),
              pa.field("fl_nascimento_apos_obito", pa.bool_()),
              pa.field("arquivo_origem", pa.string()), pa.field("coleta_origem", pa.string()),
              pa.field("tratado_em_utc", pa.string())]
    for papel in PAPEIS:
        extras.extend(pa.field(f"{c}_{papel}", pa.string()) for c in
                      ["codigo_ibge_7", "municipio_ibge", "uf_ibge", "status_vinculo"])
    schema = pa.schema([pa.field(c, pa.string()) for c in campos] + extras)
    if len(schema.names) != len(set(schema.names)):
        raise ValueError("Coluna original colide com coluna derivada.")
    agora = datetime.now(timezone.utc)
    run = agora.strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
    dados = SILVER / "dados/sim" / run
    qualidade = SILVER / "qualidade/sim" / run
    doc = SILVER / "documentacao/sim" / run
    for p in [dados, qualidade, doc]:
        p.mkdir(parents=True, exist_ok=False)
    destino = dados / "sim_2020_2024.parquet"
    parcial = destino.with_suffix(".parquet.part")
    rel = {"status": "EM_PROCESSAMENTO", "execucao": run, "registros": 0,
           "por_ano_arquivo": {}, "por_ano_obito": Counter(), "avisos": Counter(),
           "vinculos": {p: Counter() for p in PAPEIS}, "datas": {},
           "entradas": [], "referencia_ibge": str(IBGE), "sha256_ibge": sha256(IBGE),
           "colunas_originais_uniao": len(campos), "colunas_silver": len(schema),
           "colunas_ausentes_por_ano": {str(a): sorted(set(campos)-set(ns)) for a,ns in originais.items()},
           "chave_tecnica": ["ano_arquivo", "contador"], "parquet_silver": str(destino),
           "limites": "Sem deduplicação ou filtro clínico; contador não é identificador permanente. Idade e causas não decodificadas; somente datas de óbito/nascimento derivadas. Vínculos exatos, sem equivalências automáticas."}
    banco = sqlite3.connect(qualidade / "auditoria.sqlite")
    banco.execute("CREATE TABLE chaves(ano INTEGER, contador TEXT, n INTEGER, PRIMARY KEY(ano,contador))")
    banco.execute("CREATE TABLE municipios(ano INTEGER, papel TEXT, codigo TEXT, status TEXT, n INTEGER, PRIMARY KEY(ano,papel,codigo,status))")
    try:
        with pq.ParquetWriter(parcial, schema, compression="snappy") as writer:
            for ano, path in fontes.items():
                print(f"Tratando {ano}: {path.name}", flush=True)
                hash_antes = sha256(path)
                offset = 0
                with pq.ParquetFile(path) as pf:
                    for batch in pf.iter_batches(batch_size=LOTE):
                        df = batch.to_pandas().reset_index(drop=True)
                        df.columns = df.columns.str.lower()
                        # Mantém conteúdo original, inclusive espaços e códigos.
                        for c in campos:
                            if c not in df:
                                df[c] = pd.Series(pd.NA, index=df.index, dtype="string")
                        obito, so = converter_data(df.dtobito)
                        nasc, sn = converter_data(df.dtnasc)
                        df["ano_arquivo"] = ano
                        df["ano_analise"] = obito.dt.year.astype("Int16")
                        df["linha_arquivo"] = range(offset+1, offset+len(df)+1)
                        df["data_obito"] = obito.dt.date
                        df["data_nascimento"] = nasc.dt.date
                        df["status_data_obito"] = so
                        df["status_data_nascimento"] = sn
                        df["fl_obito_ano_diferente_arquivo"] = df.ano_analise.ne(ano)
                        flag = (nasc > obito).astype("boolean")
                        flag.loc[nasc.isna() | obito.isna()] = pd.NA
                        df["fl_nascimento_apos_obito"] = flag
                        for nome, series in [("obito", so), ("nascimento", sn)]:
                            rel["datas"].setdefault(nome, Counter()).update(series.value_counts().to_dict())
                        for c in ["fl_obito_ano_diferente_arquivo", "fl_nascimento_apos_obito"]:
                            rel["avisos"][c] += int(df[c].sum())
                        ids = texto(df.contador)
                        rel["avisos"]["contador_ausente"] += int(ids.isna().sum())
                        # Auditoria usa identificador aparado, conservador; não altera o original.
                        banco.executemany("INSERT INTO chaves VALUES(?,?,?) ON CONFLICT(ano,contador) DO UPDATE SET n=n+excluded.n",
                                          [(ano, str(k), int(n)) for k,n in ids.dropna().value_counts().items()])
                        rel["por_ano_obito"].update(df.ano_analise.astype("string").fillna("AUSENTE").value_counts().to_dict())
                        for papel, campo in PAPEIS.items():
                            status = vincular(df, ref, papel, campo)
                            rel["vinculos"][papel].update(status.value_counts().to_dict())
                            grp = pd.DataFrame({"ano": df.ano_analise.fillna(-1), "codigo": texto(df[campo]).fillna("<AUSENTE>"), "status": status})
                            cc = grp.groupby(["ano", "codigo", "status"], dropna=False).size()
                            banco.executemany("INSERT INTO municipios VALUES(?,?,?,?,?) ON CONFLICT(ano,papel,codigo,status) DO UPDATE SET n=n+excluded.n",
                                              [(int(a), papel, c, s, int(n)) for (a,c,s),n in cc.items()])
                        df["arquivo_origem"] = str(path)
                        df["coleta_origem"] = COLETA.name
                        df["tratado_em_utc"] = agora.isoformat()
                        writer.write_table(pa.Table.from_pandas(df[schema.names], schema=schema, preserve_index=False, safe=True))
                        offset += len(df)
                        if offset % 250_000 == 0:
                            print(f"  Processados: {offset:,} / {pf.metadata.num_rows:,}", flush=True)
                if offset != ESPERADOS[ano] or sha256(path) != hash_antes:
                    raise RuntimeError(f"{ano}: entrada mudou ou contagem divergiu.")
                rel["por_ano_arquivo"][str(ano)] = offset
                rel["registros"] += offset
                rel["entradas"].append({"arquivo": str(path), "sha256": hash_antes, "registros": offset})
                banco.commit()
                print(f"  {offset:,} registros preservados", flush=True)
        with pq.ParquetFile(parcial) as pf:
            if pf.metadata.num_rows != rel["registros"]:
                raise RuntimeError("Contagem do Parquet final divergente.")
        rel["chaves_duplicadas"] = banco.execute("SELECT COUNT(*) FROM chaves WHERE n>1").fetchone()[0]
        for nome, consulta in {
            "chaves_duplicadas.csv": "SELECT * FROM chaves WHERE n>1 ORDER BY ano,contador",
            "vinculos_municipais.csv": "SELECT * FROM municipios ORDER BY ano,papel,codigo",
            "pendencias_municipais.csv": "SELECT * FROM municipios WHERE status<>'VINCULADO_COM_POPULACAO' ORDER BY ano,papel,codigo",
        }.items():
            cursor = banco.execute(consulta)
            with (qualidade / nome).open("w", newline="", encoding="utf-8-sig") as f:
                w = csv.writer(f, delimiter=";")
                w.writerow([d[0] for d in cursor.description])
                w.writerows(cursor)
        pendencias = any(s != "VINCULADO_COM_POPULACAO" and n for cont in rel["vinculos"].values() for s,n in cont.items())
        revisar = (pendencias or rel["chaves_duplicadas"] or rel["avisos"]["contador_ausente"]
                   or rel["datas"]["obito"].get("VALIDA", 0) != rel["registros"]
                   or rel["avisos"]["fl_obito_ano_diferente_arquivo"]
                   or rel["avisos"]["fl_nascimento_apos_obito"]
                   or rel["datas"]["nascimento"].get("INVALIDA_OU_FORMATO_NAO_PREVISTO", 0))
        rel["status"] = "REVISAR_ANTES_DA_CARGA" if revisar else ("VALIDADO_COM_RESSALVAS" if rel["datas"]["nascimento"].get("AUSENTE", 0) else "VALIDADO")
        def sql_tipo(t):
            if pa.types.is_date(t): return "DATE"
            if pa.types.is_boolean(t): return "BOOLEAN"
            if pa.types.is_int16(t): return "SMALLINT"
            if pa.types.is_integer(t): return "BIGINT"
            return "TEXT"
        linhas = [f'    "{f.name}" {sql_tipo(f.type)}' for f in schema]
        linhas.append('    PRIMARY KEY ("ano_arquivo", "contador")')
        ddl = ("-- Criação de tabela nova; não executa carga nem altera tabela existente.\n"
               "-- Revisar qualidade antes da carga. Chave técnica desta coleta.\n"
               "CREATE SCHEMA IF NOT EXISTS silver;\nCREATE TABLE silver.sim (\n" + ",\n".join(linhas) + "\n);\n")
        (doc / "criar_tabela_sim.sql").write_text(ddl, encoding="utf-8")
        parcial.rename(destino)
    except Exception as e:
        rel["status"] = "ERRO"
        rel["erro"] = str(e)
        raise
    finally:
        banco.close()
        (qualidade / "relatorio_qualidade.json").write_text(json.dumps(rel, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\nStatus: {rel['status']}\nRegistros: {rel['registros']:,}\nDados: {destino}\nQualidade: {qualidade}\nSQL: {doc / 'criar_tabela_sim.sql'}")
    print("Datas:", json.dumps(rel["datas"], ensure_ascii=False))
    print("Vínculos:", json.dumps(rel["vinculos"], ensure_ascii=False))
    print("Avisos:", json.dumps(rel["avisos"], ensure_ascii=False))
    print("Chaves duplicadas:", rel["chaves_duplicadas"])
    print("Nenhuma carga foi realizada no PostgreSQL.")
    return rel


if __name__ == "__main__":
    resultado_sim = main()


Iniciando tratamento SIM — verificando a coleta selecionada...
Tratando 2020: sim_2020.parquet
  Processados: 250,000 / 1,556,824
  Processados: 500,000 / 1,556,824
  Processados: 750,000 / 1,556,824
  Processados: 1,000,000 / 1,556,824
  Processados: 1,250,000 / 1,556,824
  Processados: 1,500,000 / 1,556,824
  1,556,824 registros preservados
Tratando 2021: sim_2021.parquet
  Processados: 250,000 / 1,832,649
  Processados: 500,000 / 1,832,649
  Processados: 750,000 / 1,832,649
  Processados: 1,000,000 / 1,832,649
  Processados: 1,250,000 / 1,832,649
  Processados: 1,500,000 / 1,832,649
  Processados: 1,750,000 / 1,832,649
  1,832,649 registros preservados
Tratando 2022: sim_2022.parquet
  Processados: 250,000 / 1,544,266
  Processados: 500,000 / 1,544,266
  Processados: 750,000 / 1,544,266
  Processados: 1,000,000 / 1,544,266
  Processados: 1,250,000 / 1,544,266
  Processados: 1,500,000 / 1,544,266
  1,544,266 registros preservados
Tratando 2023: sim_2023.parquet
  Processados: 250,000

## PostgreSQL

In [6]:
from pathlib import Path
from getpass import getpass
import json

import pyarrow.parquet as pq
import psycopg
from psycopg import sql


PROJETO = Path(
    r"D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil"
)

EXECUCAO = "20261029_120603_d764c933"

ARQUIVO = (
    PROJETO / "Camada_Silver/dados/sim"
    / EXECUCAO / "sim_2020_2024.parquet"
)

QUALIDADE = (
    PROJETO / "Camada_Silver/qualidade/sim"
    / EXECUCAO / "relatorio_qualidade.json"
)

CONEXAO = {
    "host": "localhost",
    "port": 5432,
    "dbname": "observatorio_covid19_brasil",
    "user": "postgres",
    "connect_timeout": 15,
}

ANOS_ESPERADOS = {
    "2020": 1556824,
    "2021": 1832649,
    "2022": 1544266,
    "2023": 1465610,
    "2024": 1532015,
}

DISTRIBUICOES_ESPERADAS = {
    "status_data_obito": {
        "VALIDA": 7931364,
    },
    "status_data_nascimento": {
        "VALIDA": 7915211,
        "AUSENTE": 16151,
        "INVALIDA_OU_FORMATO_NAO_PREVISTO": 2,
    },
    "status_vinculo_residencia": {
        "VINCULADO_COM_POPULACAO": 7919255,
        "SEM_CORRESPONDENCIA": 12109,
    },
    "status_vinculo_ocorrencia": {
        "VINCULADO_COM_POPULACAO": 7931270,
        "SEM_CORRESPONDENCIA": 94,
    },
}

TOTAL_ESPERADO = 7_931_364
LOTE = 10_000


def carregar_sim():
    for caminho in (ARQUIVO, QUALIDADE):
        if not caminho.is_file():
            raise FileNotFoundError(caminho)

    rel = json.loads(QUALIDADE.read_text(encoding="utf-8"))

    if rel.get("execucao") != EXECUCAO:
        raise ValueError("Relatório de outra execução.")

    if rel.get("status") not in {
        "REVISAR_ANTES_DA_CARGA",
        "VALIDADO_COM_RESSALVAS",
        "VALIDADO",
    }:
        raise ValueError("A execução não terminou corretamente.")

    if rel.get("registros") != TOTAL_ESPERADO:
        raise ValueError("Total do relatório diferente do revisado.")

    if rel.get("por_ano_arquivo") != ANOS_ESPERADOS:
        raise ValueError("Contagens anuais diferentes das revisadas.")

    if rel.get("chaves_duplicadas") != 0:
        raise ValueError("Há duplicidades ou falta essa validação.")

    avisos_esperados = {
        "fl_obito_ano_diferente_arquivo": 0,
        "fl_nascimento_apos_obito": 2,
        "contador_ausente": 0,
    }

    if rel.get("avisos") != avisos_esperados:
        raise ValueError("Avisos diferentes dos revisados.")

    distribuicoes_relatorio = {
        "status_data_obito": rel["datas"]["obito"],
        "status_data_nascimento": rel["datas"]["nascimento"],
        "status_vinculo_residencia": rel["vinculos"]["residencia"],
        "status_vinculo_ocorrencia": rel["vinculos"]["ocorrencia"],
    }

    if distribuicoes_relatorio != DISTRIBUICOES_ESPERADAS:
        raise ValueError("Datas ou vínculos diferentes dos revisados.")

    tabela = sql.Identifier("silver", "sim")
    senha = getpass("Senha do PostgreSQL: ")

    with pq.ParquetFile(ARQUIVO) as parquet:
        if parquet.metadata.num_rows != TOTAL_ESPERADO:
            raise ValueError("Quantidade do Parquet divergente.")

        colunas = parquet.schema_arrow.names

        # Commit somente ao sair sem erros deste contexto.
        with psycopg.connect(**CONEXAO, password=senha) as conn:
            with conn.cursor() as cur:
                cur.execute("SELECT to_regclass('silver.sim')")
                if cur.fetchone()[0] is None:
                    raise RuntimeError(
                        "silver.sim não existe neste banco."
                    )

                cur.execute(
                    sql.SQL(
                        "LOCK TABLE {} IN SHARE ROW EXCLUSIVE MODE"
                    ).format(tabela)
                )

                cur.execute(
                    sql.SQL(
                        "SELECT EXISTS (SELECT 1 FROM {})"
                    ).format(tabela)
                )
                if cur.fetchone()[0]:
                    raise RuntimeError(
                        "silver.sim já contém dados. "
                        "Carga interrompida para evitar duplicação."
                    )

                cur.execute("""
                    SELECT column_name
                    FROM information_schema.columns
                    WHERE table_schema = 'silver'
                      AND table_name = 'sim'
                    ORDER BY ordinal_position
                """)
                colunas_banco = [r[0] for r in cur.fetchall()]

                if set(colunas_banco) != set(colunas):
                    raise RuntimeError(
                        "Colunas do banco e do Parquet divergem.\n"
                        f"Faltam no banco: "
                        f"{set(colunas) - set(colunas_banco)}\n"
                        f"Extras no banco: "
                        f"{set(colunas_banco) - set(colunas)}"
                    )

                comando = sql.SQL(
                    "COPY {} ({}) FROM STDIN"
                ).format(
                    tabela,
                    sql.SQL(", ").join(
                        sql.Identifier(c) for c in colunas
                    ),
                )

                enviados = 0
                print(
                    f"Carregando {TOTAL_ESPERADO:,} registros...",
                    flush=True,
                )

                with cur.copy(comando) as copy:
                    for lote in parquet.iter_batches(batch_size=LOTE):
                        for registro in lote.to_pylist():
                            copy.write_row(
                                tuple(registro[c] for c in colunas)
                            )

                        enviados += lote.num_rows

                        if (
                            enviados % 100_000 == 0
                            or enviados == TOTAL_ESPERADO
                        ):
                            print(
                                f"Enviados: {enviados:,} / "
                                f"{TOTAL_ESPERADO:,}",
                                flush=True,
                            )

                print("Conferindo a carga...", flush=True)

                cur.execute(
                    sql.SQL("SELECT COUNT(*) FROM {}").format(tabela)
                )
                total_banco = cur.fetchone()[0]

                if enviados != TOTAL_ESPERADO or total_banco != TOTAL_ESPERADO:
                    raise RuntimeError("Contagem total divergente.")

                for campo in ("ano_arquivo", "ano_analise"):
                    identificador = sql.Identifier(campo)
                    cur.execute(
                        sql.SQL(
                            "SELECT {}, COUNT(*) FROM {} GROUP BY {}"
                        ).format(identificador, tabela, identificador)
                    )
                    obtido = {
                        str(ano): n for ano, n in cur.fetchall()
                    }
                    if obtido != ANOS_ESPERADOS:
                        raise RuntimeError(
                            f"Contagens divergentes em {campo}."
                        )

                for campo, esperado in DISTRIBUICOES_ESPERADAS.items():
                    identificador = sql.Identifier(campo)
                    cur.execute(
                        sql.SQL(
                            "SELECT {}, COUNT(*) FROM {} GROUP BY {}"
                        ).format(identificador, tabela, identificador)
                    )
                    if dict(cur.fetchall()) != esperado:
                        raise RuntimeError(
                            f"Distribuição divergente em {campo}."
                        )

                cur.execute("""
                    SELECT
                        COUNT(*) FILTER (
                            WHERE fl_nascimento_apos_obito IS TRUE
                        ),
                        COUNT(*) FILTER (
                            WHERE data_obito IS NULL
                        ),
                        COUNT(*) FILTER (
                            WHERE data_nascimento IS NULL
                        ),
                        COUNT(*) FILTER (
                            WHERE codigo_ibge_7_residencia IS NULL
                        ),
                        COUNT(*) FILTER (
                            WHERE codigo_ibge_7_ocorrencia IS NULL
                        )
                    FROM silver.sim
                """)

                if cur.fetchone() != (2, 0, 16153, 12109, 94):
                    raise RuntimeError(
                        "Contagem de nulos ou inconsistências divergente."
                    )

    print("\nCarga concluída e confirmada!")
    print(f"Registros no banco: {total_banco:,}")
    print("Contagens anuais, datas e vínculos conferidos.")
    print("Ausências e ressalvas preservadas.")


carregar_sim()

Carregando 7,931,364 registros...
Enviados: 100,000 / 7,931,364
Enviados: 200,000 / 7,931,364
Enviados: 300,000 / 7,931,364
Enviados: 400,000 / 7,931,364
Enviados: 500,000 / 7,931,364
Enviados: 600,000 / 7,931,364
Enviados: 700,000 / 7,931,364
Enviados: 800,000 / 7,931,364
Enviados: 900,000 / 7,931,364
Enviados: 1,000,000 / 7,931,364
Enviados: 1,100,000 / 7,931,364
Enviados: 1,200,000 / 7,931,364
Enviados: 1,300,000 / 7,931,364
Enviados: 1,400,000 / 7,931,364
Enviados: 1,500,000 / 7,931,364
Enviados: 1,600,000 / 7,931,364
Enviados: 1,700,000 / 7,931,364
Enviados: 1,800,000 / 7,931,364
Enviados: 1,900,000 / 7,931,364
Enviados: 2,000,000 / 7,931,364
Enviados: 2,100,000 / 7,931,364
Enviados: 2,200,000 / 7,931,364
Enviados: 2,300,000 / 7,931,364
Enviados: 2,400,000 / 7,931,364
Enviados: 2,500,000 / 7,931,364
Enviados: 2,600,000 / 7,931,364
Enviados: 2,700,000 / 7,931,364
Enviados: 2,800,000 / 7,931,364
Enviados: 2,900,000 / 7,931,364
Enviados: 3,000,000 / 7,931,364
Enviados: 3,100,000 / 7,